# Modeling: peat condition -> fire

End-to-end scaffold for the modeling phase (see `modeling_roadmap.md`,
`decisions.md`). Flow:

1. Load the peat frame (80% histosol), restoration sites (treatment), covariates.
2. **Build matched controls** -- *you fill this in* (the causal design is yours).
3. `build_frame` -> tidy pixel-year table with a swappable fire response.
4. `fit_logit_clustered` -> `odds_ratios`.

Only elevation + histosol % are on disk today, so we match/adjust on those two
now and add distance-to-coast / land cover once they download.

In [1]:
# standard library
from pathlib import Path

# third-party
import geopandas as gpd
import numpy as np
import pandas as pd

# local (peatfire)
from peatfire import data_path, build_common_grid
from peatfire.modeling import (
    load_restoration_sites,
    available_covariates,
    covariate_on_grid,
    build_frame,
    build_modeling_grid,
    fit_logit_clustered,
    odds_ratios,
)

print("covariates on disk:", available_covariates())

covariates on disk: ['elevation', 'histosol_pct']


/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/src/peatfire/modeling/covariates.py:185: UserWarning: land_cover: directory /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/processed/land_cover/landfire_evt does not exist -- skipping.
  return [name for name, spec in COVARIATES.items() if _covariate_file(spec)]
/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/src/peatfire/modeling/covariates.py:185: UserWarning: drainage: no file matches hand_*_nc.tif in /Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/data/processed/drainage/hand -- skipping.
  return [name for name, spec in COVARIATES.items() if _covariate_file(spec)]
/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/TNC/peat_fire_stanback/src/peatfire/modeling/covariates.py:185: UserWarning: management: directory /Users

## 1. Load treatment + peat frame

In [ ]:
# Restoration polygons = the TREATMENT (reprojected to EPSG:5070; metres).
restoration_yr_col = 'End_Yr'
peat_restoration = load_restoration_sites(restoration_yr_col=restoration_yr_col) # drops rows with 0 for the restoration_yr_col and reprojects to EPSG:5070
print(peat_restoration.shape)
print(peat_restoration.columns.tolist())
peat_restoration[[restoration_yr_col, "geometry"]].head()

(5, 10)
['Id', 'acres', 'Proj_Name', 'Status_202', 'Label', 'Shape__Are', 'Shape__Len', 'Start_Yr', 'End_Yr', 'geometry']


,End_Yr,geometry
1,2023.0,"POLYGON Z ((1722161.565 1664748.186 0, 1719616..."
3,2021.0,"POLYGON Z ((1736446.599 1579988.494 0, 1736366..."
4,2019.0,"POLYGON Z ((1801397.591 1610710.885 0, 1801412..."
10,2019.0,"POLYGON Z ((1785427.126 1602486.899 0, 1786626..."
11,2019.0,"MULTIPOLYGON Z (((1799518.901 1602818.83 0, 17..."


In [3]:
# 80% peat extent as the sample frame.
aoi_nc_peat_80_histosol = gpd.read_file(data_path("processed", "peat_extent", "nc_peatlands_80_histosol_aoi.gpkg"))

aoi_nc_peat_80_histosol

,geometry
0,"MULTIPOLYGON (((1584010.262 1364375.871, 15840..."


## 2. Build matched controls  *(your step)*

This is the causal-design step deliberately left out of the `peatfire` package.
Fill in the cells below; the pointers map to the stuck points we discussed.

**a. Clip-to-drop** the restoration sites (plus a buffer) from the peat frame to
get the control *candidate* pool. `how="difference"` is the inverse of clip:

```python
BUFFER_M = 1000  # exclusion halo around restoration sites, in metres
exclusion = gpd.GeoDataFrame(geometry=[restoration.buffer(BUFFER_M).union_all()],
                             crs=restoration.crs)
candidates = gpd.overlay(aoi_nc_peat_80_histosol, exclusion, how="difference")
```

**b. Candidate pixels + nearest site.** Turn candidate peat into pixel points
(e.g. centroids of the grid cells over `candidates`), then attach each one's
nearest restoration site and distance in one call:

```python
cand_pts = gpd.sjoin_nearest(candidate_points, restoration,
                             how="left", distance_col="dist_m")
```

**c. Sample covariates** at treated and candidate pixels (reuse the grid warp):
`covariate_on_grid('elevation', grid, aoi)` and `('histosol_pct', ...)`, then
read values at the pixel locations. Assemble a table with a `treatment` 0/1
column + `elevation`, `histosol_pct`.

**d. Match** treated<->control on those covariates. Two options:
- literal nearest-neighbour on z-scored covariates: `sklearn.neighbors.NearestNeighbors`
- propensity score: `pymatch.Matcher(...).fit_scores(); .match(); .matched_data`
  (matches on P(treatment), not raw covariates -- know the difference).

Output of this section: a `units` GeoDataFrame with **`treated`** (1/0),
**`site_id`** (the matched stratum), and **`unit_id`** columns. That is all
`build_frame` needs.

In [ ]:
# TODO(you): build `units` = treated restoration polygons + matched controls,
# with columns: treated (1/0), site_id, unit_id.
#
# units = ...
raise NotImplementedError("Build the matched `units` GeoDataFrame here.")

### Balance plot (before / after matching)

The figure that *justifies* the design: overlaid distributions of elevation and
histosol % for treated vs control, before and after matching. Before, the groups
are offset; after, they should overlap -- that overlap is what lets you later
attribute a fire difference to restoration rather than geography. The
controls-next-to-restoration *map* is just a sanity check.

In [ ]:
# TODO(you): 2x2 (or 1x2) of elevation / histosol_pct, treated vs control,
# pre- and post-match. e.g. seaborn.kdeplot or overlaid hist per covariate.
# Use peatfire.set_fire_style() for consistent styling.

In [6]:
peat_restoration

,Id,acres,Proj_Name,Status_202,Label,Shape__Are,Shape__Len,Start_Yr,End_Yr,geometry
1,0,0.0,GDSNWR Pasquotank Headwaters,Completed,"Hydrologic Restoration Completed (USFWS, DOD, ...",0.004898,0.351833,2019,2023.0,"POLYGON Z ((1722161.565 1664748.186 0, 1719616..."
3,0,0.0,PLNWR Flood Resilience,Completed,Community Flood Resilience Enhancement Experim...,0.003330,0.258486,2018,2021.0,"POLYGON Z ((1736446.599 1579988.494 0, 1736366..."
4,0,0.0,ARNWR Restoration East,Completed,"Hydrologic Restoration Completed (USFWS, DOD, ...",0.004692,0.402038,2009,2019.0,"POLYGON Z ((1801397.591 1610710.885 0, 1801412..."
10,0,0.0,Dare Bombing Range WMP,Completed,"Hydrologic Restoration Completed (USFWS, DOD, ...",0.018567,0.645771,2009,2019.0,"POLYGON Z ((1785427.126 1602486.899 0, 1786626..."
11,0,0.0,ARNWR WMP,Completed,"Hydrologic Restoration Completed (USFWS, DOD, ...",0.007799,0.478279,2009,2019.0,"MULTIPOLYGON Z (((1799518.901 1602818.83 0, 17..."


## 3. Build the tidy pixel-year frame

In [ ]:
# Consumes the matched `units`. Response is swappable via `product=`.
frame = build_frame(
    units,
    product="FireCCIS311",   # swap for a severity product to model severity
    years=range(2019, 2025),
    covariate_names=None,    # default = every covariate on disk (elev, histosol),
    site_id_col='Proj_Name',
    
)
print(frame.shape)
print("burn rate by treatment:")
print(frame.groupby("treated")["burned"].mean())
frame.head()

## 4. Fit + odds ratios

In [ ]:
# Cluster-robust logistic: honest SEs (clustered on site_id), not one-per-pixel.
covs = [c for c in ("elevation", "histosol_pct") if c in frame.columns]
result = fit_logit_clustered(frame, covariates=covs)   # burned ~ treated + covs
print(result.summary())

# The headline: exp(beta) with CIs. treated < 1 => restoration lowers fire odds.
or_table = odds_ratios(result)
or_table

In [ ]:
# Prettier table for the slides (decisions/roadmap note): colour the odds ratios.
(or_table.style
    .format({"beta": "{:.3f}", "odds_ratio": "{:.2f}",
             "or_ci_low": "{:.2f}", "or_ci_high": "{:.2f}", "p_value": "{:.3f}"})
    .background_gradient(subset=["odds_ratio"], cmap="RdBu_r", vmin=0, vmax=2))

### Next

- Add a `treated:precip` interaction (dry-year effect) once climate is downloaded:
  pass `formula="burned ~ treated * precip + elevation + histosol_pct"`.
- Step up to `fit_mixed_logit` (site random intercept) to cross-check the SEs.
- Swap `product=` to a severity layer to re-run the whole thing for severity.